In [1]:
# Author: Gergely Zahoranszky-Kohalmi, PhD
#
# Email: gergely.zahoranszky-kohalmi@nih.gov
#
# Organization: National Center for Advancing Translational Sciences
#


In [2]:
# Env: syngps_rev
import syngps
from syngps import SynthGraph, AicpFunctions, graph_utils as gu
import json
import networkx as nx
import pandas as pd

import time

import os

In [3]:
DIR_IN = '../data/output/synthesis_graphs_lasm/'

FNAME_IN_ALL = []

FNAME_OUT = '../data/output/route_finding_leaves_as_sm_timing_stats_rxsmiles.tsv'

#sg_assembly_durations = []          # in seconds
#synth_route_finding_durations = []  # in seconds
#tms = []

SG_ASSEMBLY_DURATIONS = []          # in seconds
SYNTH_ROUTE_FINDING_DURATIONS = []  # in seconds
TMs = []




In [4]:
# Functions




def identify_first_synthesis_route (fname_json):
    
    first = True

    time_start = time.time()

    routes_json = None




    print (f'[*] Input file: {fname_json}')

    try:

        with open(fname_json) as f:
            routes_json = json.load(f)
            #print (f'{routes_json}')
            #print(json.dumps(routes_json, indent=4))
    
    except:
        print (f'[W] No synthesis graph found in JSON response: {fname_json}')

    #    return (None)

    if routes_json != None:
        G_sg = syngps.utils.json_to_graph(routes_json)
        #print (G_sg)

        TMID = routes_json['search_params']['target_molecule_inchikey']

        time_sg = routes_json['time_info']['synth_graph_time']
        
        

        # Generatig the artifact-free synthesis graph
        G_sg_af = gu.remove_incompatible_reactions (G_sg.copy(), target_molecule_node_id = TMID)

        # Generating combination graphs
        # and allowing to find all combination graphs based on the edge-betweenness centrality (EBC) method
        G_combinations = []
        G_combinations = gu.generate_combination_graphs (G_sg_af.copy(), method = "ebc", max_nr = 0)



        # Identifying Viable Synthesis Routes (VSRs)
        VSRs = []

        for C in G_combinations:

            R = gu.find_viable_route (C, target_module_node_id = TMID)
            
            if R[1] == 'Viable Route Candidate' or R[1] == 'Viable Synthesis Route':
                time_end = time.time()

                
                if first:
                    
                    print (f'[*] Duration (s): {time_end - time_start} , SG fetching time: {time_sg}.')
                    
                    SYNTH_ROUTE_FINDING_DURATIONS.append(time_end - time_start)
                    
                    SG_ASSEMBLY_DURATIONS.append (time_sg)
                    
                    TMs.append(TMID)

                    first = False

                #VSRs.append(R[0])
                #print (R[0])
            


    else:


        print (f'[W] No synthesis route found in JSON response: {fname_json}')

    return (None)


#####






In [5]:
search_obj = os.scandir(DIR_IN)

for dir_item in search_obj:
    if dir_item.is_file():
        
        FNAME_IN_ALL.append(DIR_IN + dir_item.name)

print(FNAME_IN_ALL)

['../data/output/synthesis_graphs_lasm/DUCJUKZQSRPECI-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/UZCLUOWLQYOFSZ-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/WTCXKDSZUHYCLH-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/FMXLKQYOBUDPPG-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/FOROUCWBDGYYOS-NSHDSACASA-N_response.json', '../data/output/synthesis_graphs_lasm/IWJOUZQQKDORMF-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/MGGGSJRWNQFCLE-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/IXZILUICVMZFAS-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/GZYNUOMXGGSXMI-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/WEEVXEKXCIBZMB-UHFFFAOYSA-N_response.json', '../data/output/synthesis_graphs_lasm/FWLFRXVPNVBXGR-CQSZACIVSA-N_response.json', '../data/output/synthesis_graphs_lasm/MRPFJQLRQGTKNI-UHFFFAOYSA-N_response.json', '../data/output

In [6]:
for fname in FNAME_IN_ALL:

    
    identify_first_synthesis_route (fname)



[*] Input file: ../data/output/synthesis_graphs_lasm/DUCJUKZQSRPECI-UHFFFAOYSA-N_response.json
[*] Duration (s): 0.028570175170898438 , SG fetching time: 0.21056032180786133.
[*] Input file: ../data/output/synthesis_graphs_lasm/UZCLUOWLQYOFSZ-UHFFFAOYSA-N_response.json
[*] Duration (s): 0.0011420249938964844 , SG fetching time: 0.01072835922241211.
[*] Input file: ../data/output/synthesis_graphs_lasm/WTCXKDSZUHYCLH-UHFFFAOYSA-N_response.json
[*] Duration (s): 0.003860950469970703 , SG fetching time: 0.020448684692382812.
[*] Input file: ../data/output/synthesis_graphs_lasm/FMXLKQYOBUDPPG-UHFFFAOYSA-N_response.json
[*] Duration (s): 0.0015940666198730469 , SG fetching time: 0.020462751388549805.
[*] Input file: ../data/output/synthesis_graphs_lasm/FOROUCWBDGYYOS-NSHDSACASA-N_response.json
[*] Duration (s): 0.004233837127685547 , SG fetching time: 0.07289767265319824.
[*] Input file: ../data/output/synthesis_graphs_lasm/IWJOUZQQKDORMF-UHFFFAOYSA-N_response.json
[*] Duration (s): 0.002306

In [7]:
df = pd.DataFrame({
    'tm': TMs,
    'route_find_time_sec': SYNTH_ROUTE_FINDING_DURATIONS,
    'synth_graph_assembly_time_sec': SG_ASSEMBLY_DURATIONS
})

print (df)


df.to_csv (FNAME_OUT, sep = '\t', index = False)



print ('[Done.]')

                              tm  route_find_time_sec  \
0    DUCJUKZQSRPECI-UHFFFAOYSA-N             0.028570   
1    UZCLUOWLQYOFSZ-UHFFFAOYSA-N             0.001142   
2    WTCXKDSZUHYCLH-UHFFFAOYSA-N             0.003861   
3    FMXLKQYOBUDPPG-UHFFFAOYSA-N             0.001594   
4    FOROUCWBDGYYOS-NSHDSACASA-N             0.004234   
..                           ...                  ...   
154  RVZKDSOXVMMGNF-UHFFFAOYSA-N             0.001401   
155  ZVERWTXKKWSSHH-UHFFFAOYSA-N             0.004669   
156  PGSOTMWBSCIDRF-UHFFFAOYSA-N             0.002482   
157  VZIQOCKMOWWQJJ-UHFFFAOYSA-N             0.004934   
158  XTXSIGYGQYVLTJ-UHFFFAOYSA-N             0.000806   

     synth_graph_assembly_time_sec  
0                         0.210560  
1                         0.010728  
2                         0.020449  
3                         0.020463  
4                         0.072898  
..                             ...  
154                       0.019762  
155                

In [8]:
# References:
#
# Ref: https://stackoverflow.com/questions/20199126/reading-json-from-a-file
# Ref: https://stackoverflow.com/questions/12943819/how-to-prettyprint-a-json-file
# Ref: Google AI Overview, 04/30/2026, by Google Gemini AI built-in Chrome
# Ref: https://www.geeksforgeeks.org/python/python-time-module/
#


